# RL Investigation: Diagnosing Seed & Goal Conditioning

Four targeted experiments to isolate whether the issue is:
1. Training seeds not producing different weights
2. Generation seeds not producing different sequences
3. RNG state converging after training
4. Goal conditioning having no effect on output

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from models.goal_directed_lstm import GoalDirectedLSTM
from assets.data_ops import load_data, encode_sequence, one_hot_encode_sequence, decode_sequence
from models import oracle

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

print('Imports OK')

In [ ]:
# Load GB1 data (same as RLExperiment._setup_gb1)
alphabet = list('ACDEFGHIKLMNPQRSTVWY')
token_to_idx = {token: idx for idx, token in enumerate(alphabet)}

df = load_data(name='gb1')
train_df = df[df['split'] == 'train'].copy()

X_train = np.array([encode_sequence(seq, token_to_idx) for seq in train_df['sequence']])
gb1_oracle = oracle.load_GB1_oracle()
y_train = gb1_oracle.score_batch(train_df['sequence'])

seq_length = len(train_df['sequence'].iloc[0])
vocab_size = len(token_to_idx)

print(f'Train sequences: {len(X_train)}, seq_length: {seq_length}, vocab_size: {vocab_size}')
print(f'y_train range: [{y_train.min():.3f}, {y_train.max():.3f}]')

In [ ]:
def train_lstm(model, X, y, epochs=20, lr=0.001, entropy_weight=0.01, verbose=False):
    """Mirror of RLExperiment._train_goal_directed_model."""
    X_tensor = torch.LongTensor(X)
    y_tensor = torch.FloatTensor(y).view(-1, 1)
    loader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=32, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.train()

    for epoch in range(epochs):
        epoch_loss = 0
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            input_seq = batch_x[:, :-1]
            target_part = batch_x[:, 1:]
            logits, _ = model(input_seq, batch_y)
            ce_loss = criterion(logits.reshape(-1, model.vocab_size), target_part.reshape(-1))
            probs = torch.softmax(logits, dim=-1)
            entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=-1).mean()
            loss = ce_loss - entropy_weight * entropy
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        if verbose and epoch % 5 == 0:
            print(f'  epoch {epoch}/{epochs} loss={epoch_loss/len(loader):.4f}')

def make_model():
    return GoalDirectedLSTM(
        vocab_size=vocab_size,
        sequence_length=seq_length,
        embedding_dim=32,
        hidden_dim=128,
    )

print('Helper functions defined')

## Step 1: Do weights actually differ across training seeds?

**Expected if seeding works**: different numbers per seed.  
**If all identical**: training seed is broken.

In [ ]:
print('=== Step 1: Weights after training with different seeds ===')
for seed in [42, 43, 44]:
    set_seed(seed)
    model = make_model()
    train_lstm(model, X_train, y_train, epochs=5)
    w = next(model.parameters()).data.flatten()[:5].tolist()
    print(f'seed={seed} weights after training: {[f"{x:.6f}" for x in w]}')

print()
print('DIAGNOSIS: If all rows are identical -> training seed is broken.')
print('           If rows differ -> training seed works correctly.')

## Step 2: Does the generation seed actually vary sequences?

Fix one trained model, vary only the `torch.manual_seed` before generation.  
**Expected**: different sequences per gen seed.  
**If identical**: `multinomial` is deterministic → model is overconfident (probabilities near 0/1).

In [ ]:
print('=== Step 2: Vary generation seed on a fixed trained model ===')

# Remove the debug print from generate() temporarily by monkey-patching
import torch.nn.functional as F

def generate_silent(model, goal_score, temperature=1.0):
    model.eval()
    input_seq = torch.zeros((1, 1), dtype=torch.long)
    goal_tensor = torch.tensor([[goal_score]], dtype=torch.float)
    generated_sequence = []
    hidden = None
    max_probs = []
    for _ in range(model.sequence_length):
        logits, hidden = model(input_seq, goal_tensor, hidden)
        next_token_logits = logits[:, -1, :] / temperature
        probs = F.softmax(next_token_logits, dim=-1)
        max_probs.append(probs.max().item())
        next_token = torch.multinomial(probs, num_samples=1)
        generated_sequence.append(next_token.item())
        input_seq = next_token
    return generated_sequence, max_probs

set_seed(42)
fixed_model = make_model()
train_lstm(fixed_model, X_train, y_train, epochs=5)

print()
for gen_seed in [42, 43, 44]:
    torch.manual_seed(gen_seed)
    results = []
    for _ in range(3):
        seq, max_probs = generate_silent(fixed_model, goal_score=1.0)
        results.append(decode_sequence(seq, alphabet))
    avg_max_prob = np.mean([p for s, mp in [generate_silent(fixed_model, 1.0)] for p in mp])
    print(f'gen_seed={gen_seed}: {results}')

print()
# Check average max probability to detect overconfidence
torch.manual_seed(42)
_, max_probs_sample = generate_silent(fixed_model, goal_score=1.0)
print(f'Avg max prob per step: {np.mean(max_probs_sample):.4f}  (1.0 = perfectly confident, ~0.05 = uniform over 20 tokens)')
print()
print('DIAGNOSIS: If seqs identical across gen_seeds -> multinomial not stochastic -> model too confident.')
print('           If seqs differ -> generation seed works.')

## Step 3: Does the RNG state converge after training?

If training consumes the RNG state in a deterministic way regardless of seed, all runs end up with the same generation RNG — meaning different seeds produce identical outputs.  
**If all hashes identical**: the original diagnosis was right — RNG converges during training.

In [ ]:
print('=== Step 3: RNG state hash after training ===')
for seed in [42, 43, 44]:
    set_seed(seed)
    model = make_model()
    train_lstm(model, X_train, y_train, epochs=5)
    state = torch.get_rng_state()
    h = hash(state.numpy().tobytes())
    print(f'seed={seed} RNG state hash after training: {h}')

print()
print('DIAGNOSIS: If all hashes identical -> RNG converges during training (fixed-length deterministic ops).')
print('           If hashes differ -> RNG state is seed-dependent after training.')

## Step 4: Does goal conditioning actually affect output?

Same RNG state, different `goal_score`. If sequences are identical the model ignores the goal input entirely.  
Also check token-level probability distributions to see how much the goal shifts them.

In [ ]:
print('=== Step 4: Goal conditioning effect ===')

set_seed(42)
model = make_model()
train_lstm(model, X_train, y_train, epochs=5)

# Same RNG seed, different goal scores
goal_scores = [0.0, 0.25, 0.5, 0.75, 1.0]
seqs = {}
for g in goal_scores:
    torch.manual_seed(42)
    seq, _ = generate_silent(model, goal_score=g)
    seqs[g] = decode_sequence(seq, alphabet)
    print(f'goal={g:.2f}: {seqs[g]}')

print()
all_same = len(set(seqs.values())) == 1
print(f'All sequences identical: {all_same}')
print()

# Deeper: compare per-step probability distributions
print('--- Per-step max probability comparison (goal=0.0 vs goal=1.0) ---')
model.eval()
probs_by_goal = {}
for g in [0.0, 1.0]:
    torch.manual_seed(42)
    input_seq = torch.zeros((1, 1), dtype=torch.long)
    goal_tensor = torch.tensor([[g]], dtype=torch.float)
    hidden = None
    step_probs = []
    for _ in range(model.sequence_length):
        logits, hidden = model(input_seq, goal_tensor, hidden)
        probs = F.softmax(logits[:, -1, :], dim=-1)
        step_probs.append(probs.detach().squeeze().numpy())
        next_token = torch.multinomial(probs, num_samples=1)
        input_seq = next_token
    probs_by_goal[g] = np.stack(step_probs)

# L1 distance per step
l1_dists = np.abs(probs_by_goal[0.0] - probs_by_goal[1.0]).sum(axis=1)
print(f'Mean L1 distance between goal=0 and goal=1 prob distributions: {l1_dists.mean():.4f}')
print(f'  (0.0 = goal has NO effect, 2.0 = completely different distributions)')
print(f'Per-step L1: {np.round(l1_dists, 3)}')

print()
print('DIAGNOSIS: If all seqs identical and L1~0 -> model completely ignores goal conditioning.')
print('           If seqs differ or L1>0.1    -> goal conditioning has measurable effect.')

## Summary

| Step | What we tested | Key metric |
|------|---------------|------------|
| 1 | Training seed → different weights? | Weight values per seed |
| 2 | Gen seed → different sequences? | Sequence diversity + avg max prob |
| 3 | RNG state converges after training? | RNG state hash |
| 4 | Goal score changes output? | L1 distance between prob distributions |